
# Warehouse Throughput Prediction

**Goal:** Predict hourly warehouse throughput using operational variables such as staffing, order volume, automation utilization, downtime, absenteeism, and workload characteristics.

> **Data note:** This project uses a **synthetic dataset** created to resemble realistic warehouse operations. It does not contain proprietary or employer data.

### Tools
Python, pandas, scikit-learn, matplotlib

### Modeling approach
This notebook:
1. Loads and inspects warehouse operations data.
2. Performs basic exploratory analysis.
3. Preprocesses numerical and categorical variables.
4. Trains Linear Regression, Random Forest, and Gradient Boosting models.
5. Compares MAE, RMSE, and R².
6. Evaluates the strongest model and identifies the most influential operational variables.

**Best model in this run:** Linear Regression  
**MAE:** 6.22 units/hour  
**RMSE:** 7.77 units/hour  
**R²:** 0.828


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance


In [ ]:

df = pd.read_csv("warehouse_operations_synthetic.csv")
df.head()


In [ ]:

df.info()


In [ ]:

df.describe(numeric_only=True).T


## Exploratory Analysis

In [ ]:

shift_summary = (
    df.groupby("shift")["throughput_units_per_hour"]
      .agg(["mean", "median", "count"])
      .sort_values("mean", ascending=False)
)
shift_summary


In [ ]:

plt.figure(figsize=(6, 4.5))
plt.bar(shift_summary.index, shift_summary["mean"])
plt.xlabel("Shift")
plt.ylabel("Average Throughput (units/hour)")
plt.title("Average Throughput by Shift")
plt.tight_layout()
plt.show()


In [ ]:

numeric_cols = df.select_dtypes(include=np.number).columns
correlations = (
    df[numeric_cols]
    .corr()["throughput_units_per_hour"]
    .sort_values(ascending=False)
)
correlations


## Model Preparation

In [ ]:

target = "throughput_units_per_hour"

X = df.drop(columns=[target])
y = df[target]

categorical = ["day_of_week", "shift"]
numeric = [c for c in X.columns if c not in categorical]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)


In [ ]:

preprocess_linear = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

preprocess_tree = ColumnTransformer([
    ("num", "passthrough", numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

models = {
    "Linear Regression": Pipeline([
        ("prep", preprocess_linear),
        ("model", LinearRegression())
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocess_tree),
        ("model", RandomForestRegressor(
            n_estimators=250,
            random_state=42,
            min_samples_leaf=2,
            n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("prep", preprocess_tree),
        ("model", GradientBoostingRegressor(random_state=42))
    ]),
}


## Train and Compare Models

In [ ]:

results = []
trained_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    trained_models[name] = model
    predictions[name] = pred

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df



### How to read the metrics
- **MAE:** Average size of the prediction error in units/hour. Lower is better.
- **RMSE:** Similar to MAE, but penalizes larger errors more strongly. Lower is better.
- **R²:** Fraction of variation explained by the model. Higher is better.


In [ ]:

best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
best_pred = predictions[best_name]

best_name


## Predicted vs. Actual Throughput

In [ ]:

plt.figure(figsize=(7, 5))
plt.scatter(y_test, best_pred, alpha=0.55)

low = min(y_test.min(), best_pred.min())
high = max(y_test.max(), best_pred.max())
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Actual Throughput (units/hour)")
plt.ylabel("Predicted Throughput (units/hour)")
plt.title(f"Predicted vs. Actual Throughput — {best_name}")
plt.tight_layout()
plt.show()


## Operational Driver Analysis

In [ ]:

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=12,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False)

importance_df


In [ ]:

top_imp = importance_df.head(10).sort_values("Importance")

plt.figure(figsize=(8, 5.5))
plt.barh(top_imp["Feature"], top_imp["Importance"])
plt.xlabel("Permutation Importance")
plt.ylabel("Feature")
plt.title(f"Top Feature Drivers — {best_name}")
plt.tight_layout()
plt.show()



## Key Findings

For this synthetic dataset:

- **Linear Regression** achieved the lowest test RMSE.
- The model's test MAE was approximately **6.2 units/hour**.
- The model explained approximately **82.8%** of the variation in throughput on the held-out test set.
- The five most influential model inputs were: **staff_count, order_volume, equipment_downtime_min, shift, automation_utilization_pct**.
- The results demonstrate how operational data can be used to support staffing, capacity planning, and process-improvement decisions.

Because the dataset is synthetic, these findings demonstrate the modeling workflow rather than making claims about a real warehouse.



## Possible Extensions

- Replace the synthetic data with a public logistics or manufacturing dataset.
- Add time-series demand forecasting.
- Optimize staffing levels subject to service-level constraints.
- Build a Streamlit dashboard for interactive throughput scenarios.
- Add SHAP explanations for model interpretability.
